## Module 1: DGPs
#### Create simulation engines that generate observational datasets with known ground-truth ATE.

*   Used as benchmark to rigorously evaluate whether ICL instructions (and few-shot examples) enable a general-purpose pre-trained LLM to produce accurate, robust, and appropriately uncertain ATE estimates.
*   Performance is assessed against simulation ground truth and compared to classical estimators.
*   Apply findings to complementary case study that uses PSID data.



General assumptions:

*   Covariates X are p-dimensional, independent, real numbers ~ N(0, I)
*   ε ~ N(0, σ^2)




In [ ]:
# Import packages
import numpy as np
import random
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define the logistic function
def sigm(t):
    return 1/(1 + np.exp(-t))

In [ ]:
# Fixed seed
from numpy.random import default_rng
rng = default_rng(123)

In [ ]:
# Fixed p (# covariates) and n (# observations)
p = 200
n = 1000

In [ ]:
# Initialize w and b as random draw from uniform distribution (to be normalized)
w = rng.uniform(0,1,size=(p,1))
b = rng.uniform(0,1,size=(p,1))

In [ ]:
# Generate n observations of p-dimensional covariates; sigma = covariance matrix
def gen_covariate(p,n,sigma):
  return rng.multivariate_normal(
      mean = np.zeros(p),
      cov = sigma,
      size = n
  ).T

sigma = np.eye(p) # Identity matrix
X = gen_covariate(p,n,sigma) # X = pxn matrix, where each column is an observation

In [ ]:
# Generate model error ~ N(0, std^2): epsilon = 1xn
def gen_esp(n, std):
  return rng.normal(loc = 0.0, scale = std, size = (1,n)) # each element or column are independent Gaussian draws

std = 1
esp = gen_esp(n, std)

In [ ]:
# Saves data D as csv, with columns for covariates X1,...,Xp, A, Y_0, Y_1, Y and n rows of observations
def save_data(D, filename):
  X, A, Y_0, Y_1, Y = D

  X = X.T # nxp

  #nx1
  A = A.reshape(-1,1)
  Y_0 = Y_0.reshape(-1,1)
  Y_1 = Y_1.reshape(-1,1)
  Y = Y.reshape(-1,1)

  # Horizontally combine
  full_data = np.hstack([X, A, Y_0, Y_1, Y])

  # Write column names
  columns = ([f"X{i+1}" for i in range(X.shape[1])] + ["A", "Y_0", "Y_1", "Y"])

  df = pd.DataFrame(full_data, columns = columns)
  df.to_csv(f"/content/drive/My Drive/mps_dgp/{filename}",index = False) # saves to Google Drive folder called mps_dgp

### Engine #1
Linear baseline with tunable confounding (analytical ATE)

In [ ]:
# X = pxn covariates; w = px1
def gen_treat_1(X, w, a, k):
  w = w / np.linalg.norm(w) # normalize to unit vector
  w = w.reshape(-1,1)
  b = a + k * (w.T @ X) # b = 1xn
  e = sigm(b) # logit function applied element-wise to b
  A = rng.binomial(1,e) # n independent Bernoulli(e[i]) draws
  return A # A = 1xn

# b = px1 vector; t = ground truth ATE, esp = model error 1xn
def gen_outcome_1(X, A, b, t, esp):
  b = b / np.linalg.norm(b) # normalize to unit vector
  b = b.reshape(-1,1) # Ensures b = px1
  Y_0 = b.T @ X + esp
  Y_1 = b.T @ X + t + esp
  Y = b.T @ X + t * A + esp
  return Y_0, Y_1, Y # all 1xn outputs

In [ ]:
def main_1(X, w, a, k, b, t, esp):
  A = gen_treat_1(X, w, a, k)
  Y_0, Y_1, Y = gen_outcome_1(X, A, b, t, esp)
  D = (X, A, Y_0, Y_1, Y)
  return D

In [ ]:
a = 1
k = 1
t = [0,0.5,1]
for tau in t:
  D = main_1(X, w, a, k, b, tau, esp)
  save_data(D, f"Engine_1_tau_{tau}.csv")

KeyboardInterrupt: 

### Engine #2
Nonlinear outcome with constant effect (analytical ATE)

In [ ]:
# X = pxn covariates; w = px1
def gen_treat_2(X, w, a, k):
  w = w / np.linalg.norm(w) # normalize to unit vector
  w = w.reshape(-1,1)
  b = a + k * (w.T @ X) # b = 1xn
  e = sigm(b) # logit function applied element-wise to b
  A = rng.binomial(1,e) # n independent Bernoulli(e[i]) draws
  return A # A = 1xn

# b = px1 vector; t = ground truth ATE, esp = model error 1xn
def gen_outcome_2(X, A, t, esp):

  if X.shape[0] < 5:
        raise ValueError("Engine 2 requires p >= 5 because it uses X1..X5.")
  x1 = X[0,:]
  x2 = X[1,:]
  x3 = X[2,:]
  x4 = X[3,:]
  x5 = X[4,:]
  g = (0.5*x1**2) + np.sin(x2) + (0.25*x3*x4) - (0.1*x5**3) # 1xn vector

  Y_0 = g + esp
  Y_1 = g + t + esp
  Y = g + t * A + esp
  return Y_0, Y_1, Y # all 1xn outputs

In [ ]:
def main_2(X, w, a, k, t, esp):
  A = gen_treat_2(X, w, a, k)
  Y_0, Y_1, Y = gen_outcome_2(X, A, t, esp)
  D = (X, A, Y_0, Y_1, Y)
  return D

In [ ]:
a = 1
k = 1
tau3 = np.max(np.abs(X[4, :]**3))
tau2 = tau3/2
t = [0,tau2,tau3]
for tau in t:
  D = main_2(X, w, a, k, tau, esp)
  save_data(D, f"Engine_2_tau_{round(tau,2)}.csv")

### Engine #3
Interaction-heavy HTE (analytic ATE under Gaussian X)

In [ ]:
# p = number of covariates; n = number of observations
def gen_covar_3(p, n, rng = None):
    if rng is None:
        rng = np.random.default_rng()
    return rng.normal(0.0, 1.0, size=(n, p))

# X = covariates; v = a (possibly different) unit vector than w
def gen_treat_3(X, v, alpha, kappa, rng = None):
    if rng is None:
        rng = np.random.default_rng()
    v = np.asarray(v, float)
    v = v / np.linalg.norm(v)
    score = alpha + kappa * (X @ v)   # (n,)
    p = sigm(score)
    A = rng.binomial(1, p, size = X.shape[0]).astype(int)
    return A

$ m_0(X) = 0.2X_1 + 0.2X_2 + 0.2 X_3 X_4 $

In [ ]:
def m0_engine3(X):
    if X.shape[1] < 4:
        raise ValueError()
    return 0.2 * X[:, 0] + 0.2 * X[:, 1] + 0.2 * (X[:, 2] * X[:, 3])

$\tau(X) = \tau_0 + \tau_1 X_1 + \tau_2 X_2^2$

In [ ]:
def tau_engine3(X, tau0, tau1, tau2):
    if X.shape[1] < 3:
        raise ValueError()
    return tau0 + tau1 * X[:, 0] + tau2 * (X[:, 2]**2 - 1.0)

In [ ]:
# b = p-dimensional vector of real numbers; t = ground truth ATE, sd = standard deviation of model error
def gen_outcome_3(X, A, tau0, tau1, tau2, sd, rng = None):
    if rng is None:
        rng = np.random.default_rng()
    eps = rng.normal(0.0, sd, size = X.shape[0])
    base = m0_engine3(X)
    tau_x = tau_engine3(X, tau0, tau1, tau2)
    Y_0 = base + eps
    Y_1 = base + tau_x + eps
    Y = Y_0 * (1 - A) + Y_1 * A
    return Y_0, Y_1, Y

In [ ]:
def main_3(p, n, v, alpha, kappa, tau0, tau1, tau2, sd, seed = 0):
    rng = np.random.default_rng(seed)
    X = gen_covar_3(p, n, rng = rng)
    A = gen_treat_3(X, v, alpha, kappa, rng = rng)
    Y_0, Y_1, Y = gen_outcome_3(X, A, tau0, tau1, tau2, sd, rng = rng)
    return (X.T, A, Y_0, Y_1, Y)

In [ ]:
v = np.random.default_rng(0).normal(size = p)

alpha = 1
kappa = 1
sd = 1

tau_list = [
    (0, 0, 0),
    (0, 1, 4),
    (0, 0, 4),
    (1, 1, 1),
    (2, 4, 6),
]

for (tau0, tau1, tau2) in tau_list:
    D = main_3(p, n, v, alpha, kappa, tau0, tau1, tau2, sd, seed = 0)
    save_data(D, f"Engine_3_tau0_{tau0}_tau1_{tau1}_tau2_{tau2}.csv")

### Engine #4
High-dimensional sparse confounding + HTE (analytic ATE)


In [ ]:
def gen_treat_4(X, SA, w, a, k):
    """
    e(X)=sigm(a + k * sum_{j in SA} w_j X_j),  A~Bernoulli(e)
    SA: 0-based index list OR indicator vector (len p). Uses only SA (<5).
    X: p×n, w: p×1 or p,
    returns A: 1×n
    """
    p, n = X.shape
    SA = np.asarray(SA) if SA is not None else np.array([], dtype=int)
    SA = (np.where(SA.astype(bool))[0] if (SA.size == p and SA.ndim == 1) else SA.astype(int).ravel())
    SA = SA[SA < min(5, p)]  # SA < 5

    w = np.asarray(w, float).reshape(-1, 1)  # no need to normalize; only w[SA] used

    b_lin = a if SA.size == 0 else a + k * (w[SA].T @ X[SA])   # 1×n
    e = sigm(b_lin)                                           # 1×n
    A = rng.binomial(1, e).astype(int)                        # 1×n
    return A
def gen_outcome_4(X, A, SY, b, t0, t1, j_star, esp):
    """
    m0(X)=sum_{j in SY} b_j X_j, tau(X)=t0+t1*X_{j_star}
    Y0=m0+esp, Y1=m0+tau+esp, Y=m0 + A*tau + esp
    SY: same format as SA, uses only SY (<5). Vectorized (no loop).
    returns (Y_0,Y_1,Y): all 1×n
    """
    p, n = X.shape
    SY = np.asarray(SY) if SY is not None else np.array([], dtype=int)
    SY = (np.where(SY.astype(bool))[0] if (SY.size == p and SY.ndim == 1) else SY.astype(int).ravel())
    SY = SY[SY < min(5, p)]  # SY < 5

    b = np.asarray(b, float).reshape(-1, 1)

    m0 = np.zeros((1, n)) if SY.size == 0 else (b[SY].T @ X[SY])  # 1×n
    tau_x = t0 + t1 * X[int(j_star):int(j_star)+1, :]             # 1×n

    Y_0 = m0 + esp
    Y_1 = m0 + tau_x + esp
    Y   = m0 + A * tau_x + esp
    return Y_0, Y_1, Y

In [ ]:
def main_4(X, SA, SY, w, a, k, b, t0, t1, j_star, esp):
    """
    Returns D=(X,A,Y_0,Y_1,Y) exactly like engine1.
    """
    A = gen_treat_4(X, SA, w, a, k)
    Y_0, Y_1, Y = gen_outcome_4(X, A, SY, b, t0, t1, j_star, esp)
    return (X, A, Y_0, Y_1, Y)

taus = [(0,0), (0,1), (1,1), (1,4)]

a = 1
k = 1
j_star = 1

SA = [0,1,2]
SY = [0,3,4]

for (t0, t1) in taus:
    D = main_4(X, SA, SY, w, a, k, b, t0, t1, j_star, esp)
    save_data(D, f"Engine_4_tau0_{t0}_tau1_{t1}.csv")

### Engine #5
Overlap/positivity near-violations (explicit overlap floor)

In [ ]:
# X = pxn covariates; w = px1
# ep = overlap grid {0.10, 0.05, 0.02, 0.01, 0.005}
def gen_treat_5(X, w, a, k, ep):
  w = w / np.linalg.norm(w) # normalize to unit vector
  w = w.reshape(-1,1)
  b = a + k * (w.T @ X) # b = 1xn
  e = sigm(b) # logit function applied element-wise to b
  e = ep + ((1-(2*ep))*e) # stress test
  A = rng.binomial(1,e) # n independent Bernoulli(e[i]) draws
  return A # A = 1xn

In [ ]:
def main_5(X, w, a, k, ep, t, engine, b = None):
  A = gen_treat_5(X, w, a, k, ep)
  if engine not in ["Engine 1", "Engine 2", "Engine 3", "Engine 4"]:
    raise ValueError("Please enter a valid engine: Engine 1, Engine 2, Engine 3, or Engine 4")
  if engine == "Engine 1":
    if b is None:
        raise ValueError("Engine 1 requires parameter b (px1).")
    Y_0, Y_1, Y = gen_outcome_1(X, A, b, t, esp)
  if engine == "Engine 2":
    Y_0, Y_1, Y = gen_outcome_2(X, A, t, esp)

  if engine == "Engine 3":
    tau0, tau1, tau2 = t
    X_np = X.T
    A_n = np.array(A).reshape(-1)
    sd = float(np.std(esp))
    Y_0, Y_1, Y = gen_outcome_3(X_np, A_n, tau0, tau1, tau2, sd, rng = rng)

  if engine == "Engine 4":
    # requires: b (px1), SY, t as (t0,t1), j_star
    if b is None:
        raise ValueError("Engine 4 requires parameter b (px1).")
    if "SY" not in globals():
        raise ValueError("Engine 4 requires SY (sparse outcome index set).")
    if "j_star" not in globals():
        raise ValueError("Engine 4 requires j_star (HTE driver index).")

    # t can be scalar in your loops; for engine4 we need (t0,t1)
    # so assume you pass t as a tuple (t0,t1) when engine == "Engine 4"
    try:
        t0, t1 = t
    except Exception:
        raise ValueError("Engine 4 needs t=(t0,t1) tuple, e.g. (0,1),(1,4).")

    Y_0, Y_1, Y = gen_outcome_4(X, A, SY, b, t0, t1, j_star, esp)

  D = (X, A, Y_0, Y_1, Y)
  return D

In [ ]:
# Engine 1
engine = "Engine 1"
a = 1
k = 1
t = [0,0.5,1]
eps = [0.10, 0.05, 0.02, 0.01, 0.005]
for tau in t:
  for ep in eps:
    D = main_5(X, w, a, k, ep, tau, engine, b)
    save_data(D, f"Engine_5_Outcome_{engine}_tau_{round(tau,2)}_overlap_{ep}.csv")

In [ ]:
# Engine 2
engine = "Engine 2"
a = 1
k = 1
tau3 = np.max(np.abs(X[4, :]**3))
tau2 = tau3/2
t = [0,tau2,tau3]
eps = [0.10, 0.05, 0.02, 0.01, 0.005]
for tau in t:
  for ep in eps:
    D = main_5(X, w, a, k, ep, tau, engine)
    save_data(D, f"Engine_5_Outcome_{engine}_tau_{round(tau,2)}_overlap_{ep}.csv")

In [ ]:
# Engine 3
engine = 'Engine 3'
a = 1
k = 1
eps = [0.10, 0.05, 0.02, 0.01, 0.005]

tau_list = [
    (0, 0, 0),
    (0, 1, 4),
    (0, 0, 4),
    (1, 1, 1),
    (2, 4, 6),
]

for tau in tau_list:
  for ep in eps:
    D = main_5(X, w, a, k, ep, tau, engine)
    save_data(D, f'Engine_5_Outcome_{engine}_tau0_{tau[0]}_tau1_{tau[1]}_tau2_{tau[2]}_overlap_{ep}.csv')

In [ ]:
# Engine 4
engine = 'Engine 4'
a = 1
k = 1
SY = [0, 3, 4]
j_star = 1
taus = [(0, 0), (0, 1), (1, 1), (1, 4)]
eps = [0.10, 0.05, 0.02, 0.01, 0.005]
for t in taus:
  for ep in eps:
    D = main_5(X, w, a, k, ep, t, engine, b)   # note: t is tuple here
    save_data(D, f"Engine_5_Outcome_{engine}_tau0_{t[0]}_tau1_{t[1]}_overlap_{ep}.csv")

### Engine #6
Nonlinear treatment assignment (nonlinear confounding)

- $X$ denote the covariate vector  
- $a_0$ denote the intercept ($\alpha_0$)  
- $k$ denote the confounding strength ($\kappa$)

#### Define the nonlinear score function:
$$
s(X) = X_1 + X_2^2 - X_3 X_4 + 0.5 \sin(X_5)
$$

The propensity score is given by:
$$
e(X) = \sigma\left(a_0 + k \cdot s(X)\right)
$$
where
$$
\sigma(t) = \frac{1}{1 + e^{-t}}
$$

The treatment indicator is generated as:
$$
A \mid X \sim \text{Bernoulli}(e(X))
$$

#### Define the baseline outcome function:
$$
m_0(X) = 0.5 X_1 + 0.25 X_2^2 - 0.2 X_3 X_4
$$

The potential outcomes are given by:
$$
Y(0) = m_0(X) + \varepsilon
$$
$$
Y(1) = m_0(X) + \tau_0 + \varepsilon
$$

where the noise term satisfies:
$$
\varepsilon \sim \mathcal{N}(0, \sigma^2)
$$

The observed outcome is:
$$
Y = m_0(X) + A \tau_0 + \varepsilon
$$

The true average treatment effect (ATE) is:
$$
\text{ATE} = \mathbb{E}[Y(1) - Y(0)] = \tau_0
$$

In [ ]:
# p = number of covariates (must be >= 5); n = number of observations
def gen_covar_6(p, n):
    return np.random.normal(0, 1, size=(n, p))

def gen_treat_6(X, a0, k):
    if X.shape[1] < 5:
        raise ValueError("Engine 6 requires p >= 5 because it uses X1..X5.")

    x1 = X[:, 0]
    x2 = X[:, 1]
    x3 = X[:, 2]
    x4 = X[:, 3]
    x5 = X[:, 4]

    s = x1 + (x2 ** 2) - (x3 * x4) + 0.5 * np.sin(x5)
    e = sigm(a0 + k * s)
    A = np.random.binomial(1, e, size=X.shape[0])

    return A, e

def gen_outcome_6(X, A, tau0, sd):
    if X.shape[1] < 4:
        raise ValueError("Engine 6 outcome needs at least p >= 4 (uses X1..X4).")

    x1 = X[:, 0]
    x2 = X[:, 1]
    x3 = X[:, 2]
    x4 = X[:, 3]

    m0 = 0.5 * x1 + 0.25 * (x2 ** 2) - 0.2 * (x3 * x4)
    eps = np.random.normal(0.0, sd, size=X.shape[0])
    Y = m0 + A * tau0 + eps

    return Y

def main_6(p, n, a0, k, tau0, sd, seed=0, return_propensity=True):
    rng = np.random.default_rng(seed)
    np.random.seed(seed)

    X = gen_covar_6(p, n)
    A, e = gen_treat_6(X, a0, k)
    Y = gen_outcome_6(X, A, tau0, sd)

    D = (X, A, Y)
    if return_propensity:
        return D, e
    return D

### Engine #7


In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))
def engine7_mcar(X, A, Y, pi, seed=None):
    """
    MCAR: R ~ Bernoulli(pi) independent of (X, A, Y)
    Observe Y only when R=1.
    """
    rng = np.random.default_rng(seed)
    n = len(Y)
    R = rng.binomial(1, pi, size=n).astype(int)
    Y_obs = Y.astype(float).copy()
    Y_obs[R == 0] = np.nan
    return (X, A, Y_obs, R)

def engine7_mar(X, A, Y, gamma0, gammaA, gamma, seed=None):
    """
    MAR: P(R=1 | X, A) = sigmoid(gamma0 + gammaA*A + gamma^T X)
    Observe Y only when R=1.
    """
    rng = np.random.default_rng(seed)
    n = len(Y)
    gamma = np.asarray(gamma)

    p_obs = np.zeros(n, dtype=float)
    for i in range(n):
        score = gamma0 + gammaA * A[i] + float(np.dot(gamma, X[i]))
        p_obs[i] = sigmoid(score)

    R = rng.binomial(1, p_obs, size=n).astype(int)
    Y_obs = Y.astype(float).copy()
    Y_obs[R == 0] = np.nan
    return (X, A, Y_obs, R)





### Engine #8
Measurement error + heavy-tailed noise

- (1)Treatment assignment and outcomes depend on TRUE covariates X
- (2)Observed covariates W are noisy measurements of X (W = X + U)
- (3)Outcome noise epsilon follows a mixture distribution

In [ ]:
from numpy.random import default_rng

def sigm(t):
    return 1/(1 + np.exp(-t))

def mixed_normal_noise(n, sd, w_main=0.95, sd_multiplier=10.0, rng=None):
    if rng is None:
        rng = default_rng()
    main = rng.random(n) < w_main
    eps = rng.normal(0.0, sd, size=n)
    tail_n = (~main).sum()
    if tail_n > 0:
        eps[~main] = rng.normal(0.0, sd_multiplier*sd, size=tail_n)
    return eps.reshape(1, n)

In [ ]:
def main_8(X, w, a, k, b, tau, sd, sigma_u=0.5, w_main=0.95, sd_multiplier=10.0, rng=None):
    """
    # X = p x n matrix of latent covariates
    # p = number of covariates
    # n = number of observations
    # w = p-dimensional vector for treatment assignment (confounding)
    # a = intercept term in the treatment assignment model
    # k = confounding strength in the treatment assignment model
    # b = p-dimensional vector of outcome coefficients
    # tau = ground-truth average treatment effect
    # sd = standard deviation of baseline outcome noise
    # sigma_u = standard deviation of measurement error in covariates (W = X + U)
    # w_main = probability that outcome noise is drawn from the main Gaussian component
    # sd_multiplier = scale multiplier for the heavy-tailed noise component
    # rng = random number generator
    """
    if rng is None:
        rng = default_rng()

    p, n = X.shape

    w = np.asarray(w, float).reshape(p, 1)
    b = np.asarray(b, float).reshape(p, 1)

    # normalize
    w = w / np.linalg.norm(w)
    b = b / np.linalg.norm(b)

    # A | X
    score = a + k * (w.T @ X)
    e = sigm(score)
    A = rng.binomial(1, e).astype(int)

    # measurement error
    U = rng.normal(0.0, sigma_u, size=(p, n))
    W = X + U

    # heavy-tailed eps
    eps = mixed_normal_noise(n, sd, w_main=w_main, sd_multiplier=sd_multiplier, rng=rng)

    # outcome uses W
    mu = b.T @ W
    Y0 = mu + eps
    Y1 = mu + tau + eps
    Y  = Y0*(1-A) + Y1*A

    return (W, A, Y0, Y1, Y)

In [ ]:
sigma_u = 0.5
w_main = 0.95
sd_multiplier = 10.0
a = 1
k = 1
sd = 1

tau_list = [0, 2, 4, 6]
for tau in tau_list:
    D8 = main_8(X, w, a, k, b, tau, sd,
                sigma_u=sigma_u, w_main=w_main, sd_multiplier=sd_multiplier, rng=rng)
    save_data(D8, f"Engine_8_tau_{tau}.csv")

## Module 1: Data-to-Text
####



In [ ]:
# Small n (<200)

def small_to_text(csv_path):
  df = pd.read_csv(csv_path)
  print("DATA_START")

  print(" ".join(df.columns))

  for _, row in df.iterrows():
    print(" ".join(map(str, row.values)))

  print("DATA_END")

In [ ]:
# Large n

def large_to_subset(csv_path, subset_n):
  df = pd.read_csv(csv_path)
  print("SUBSETTED_DATA_START")

  print(" ".join(df.columns))

  treated = df[df['A'] == 1].sample(n=int(subset_n/2), replace=False)
  untreated = df[df['A'] == 0].sample(n=int(subset_n/2), replace=False)
  subset = pd.concat([treated, untreated], axis = 0)

  for _, row in subset.iterrows():
    print(" ".join(map(str, row.values)))

  print("SUBSETTED_DATA_END")

def summary_stats(csv_path):
  df = pd.read_csv(csv_path)
  results = []

  x_cols = [c for c in df.columns if c.startswith('X')]
  for col in x_cols:
    treated = df[df['A'] == 1][col].dropna()
    control = df[df['A'] == 0][col].dropna()
    if len(treated) == 0 or len(control) == 0:
      continue

    # Treated-group mean
    mean_A1 = treated.mean()
    mean_A0 = control.mean()

    # Treated-group sd
    sd_A1 = treated.std(ddof = 1)
    sd_A0 = control.std(ddof = 1)

    # Standardized mean difference (SMDs)
    pooled_sd = np.sqrt(0.5 * (sd_A1 ** 2 + sd_A0 ** 2))
    if pooled_sd == 0:
      smd = 0.0
    else:
      smd = (mean_A1 - mean_A0) / pooled_sd

    results.append((col, mean_A0, mean_A1, sd_A0, sd_A1, smd))

  return results # list of tuples for each column (col, summary statistics)

In [ ]:
# precleaning of the data
def preclean(csv_path):
  df = pd.read_csv(csv_path)
  x_prefix = 'X'
  decimals = 3
  standardize_x = False
  row_shuffle_seed = None

  # sort the colmuns for X
  x_cols = [c for c in df.columns if c.startswith(x_prefix)]
  def x_key(name: str):
    suffix = name[len(x_prefix): ]
    if suffix.isdigit():
      return int(suffix)
    else:
      return suffix
  x_cols = sorted(x_cols, key = x_key) # sort as X1, X2, ..., Xp

  # column ordering follows A, Y, X1, X2, ..., Xp
  # drop Y0, Y1
  keep = [c for c in (['A', 'Y'] + x_cols) if c in df.columns]
  df = df[keep]

  # explicit missingness tokens
  # if a value is missing, print NaN
  for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors = 'coerce')
  df = df.replace([np.inf, -np.inf], np.nan)

  # row order randomization for position bias control
  if row_shuffle_seed is not None:
    df = df.sample(frac = 1.0, random_state = row_shuffle_seed).reset_index(drop = True)

  # enforce A in {0, 1}
  if 'A' in df.columns:
    df['A'] = df['A'].round().astype('Int64')
    df['A'] = df['A'].clip(lower = 0, upper = 1)

  # scale management
  # standardize covariates (mean 0, variance 1)
  if standardize_x and len(x_cols) > 0:
    x_present = [c for c in x_cols if c in df.columns]
    means = df[x_present].mean(skipna = True)
    stds = df[x_present].std(skipna = True, ddof = 0)
    stds = stds.replace(0, np.nan) # avoid division by zero
    df[x_present] = (df[x_present] - means) / stds
    df[x_present] = df[x_present].fillna(0.0)

  # standardize numeric formatting
  for c in df.columns:
    if c != 'A':
      df[c] = df[c].round(decimals)

  return df


In [ ]:
# make output machine-gradable
import json
def build_prompt(serialized_data):
  # task instructions
  instructions = []
  instructions.append('TASK_INSTRUCTIONS')
  instructions.append(
      'You are given observational data with covariates X, treatment A (0/1), and outcome Y.'
  )
  instructions.append(
      'Estimate the average treatment effect (ATE) from observational data (X, A, Y).'
  )
  instructions.append(
      'Assume unconfoundedness given X and discuss overlap/positivity.'
  )
  instructions.append(
      'Provide a point estimate and a 95% interval, with warnings if overlap is weak.'
  )
  instructions.append('END_TASK_INSTRUCTIONS')

  # few-shot examples (list of dictionaries)
  examples_block = []
  few_shot_examples = None
  if few_shot_examples:
    examples_block.append('FEW_SHOT_EXAMPLES_START')
    for i, ex in enumerate(few_shot_examples, start = 1):
      examples_block.append(f'EXAMPLE_{i}_DATASET_START')
      examples_block.append(ex['dataset'].strip())
      examples_block.append(f'EXAMPLE_{i}_DATASET_END')
      examples_block.append(f'EXAMPLE_{i}_ANSWER_START')
      examples_block.append(json.dumps(ex['answer_json'], ensure_ascii = False))
      examples_block.append(f'EXAMPLE_{i}_ANSWER_END')
    examples_block.append('FEW_SHOT_EXAMPLES_END')

  # the evaluation dataset
  eval_block = []
  eval_block.append('serialized_data.strip')

  # output schema reminder
  schema_block = []
  include_schema_reminder = True
  if include_schema_reminder:
    schema_block.append('OUTPUT_SCHEMA')
    schema_block.append('Return exactly ONE JSON object with these required fields:')
    schema_block.append('{')
    schema_block.append('   "ate_hat": <number>,')
    schema_block.append('   "ci95": [<lower>, <upper>],')
    schema_block.append('   "method": "<short label>"')
    schema_block.append('   "assumptions": ["unconfoundedness_given_X", "overlap"],')
    schema_block.append('   "overlap_warning": <true/false>,')
    schema_block.append('   "notes": "<brief justification, <= 120 words>"')
    schema_block.append('}')
    schema_block.append('Rules:')
    schema_block.append('The interval must satisfy ci95[0] <= ate_hat <= ci95[1].')
    schema_block.append(
        'If overlap is weak (as suggested by propensity quantiles/bins or extreme imbalance), the model should set overlap_warning = true and widen the interval.'
    )
    schema_block.append(
        'Method must be a concise descriptor(e.g., "AIPW: logistic PS + linear outcome", "IPW(trimmed)", "outcome regression").'
    )
    schema_block.append('END_OUTPUT_SCHEMA')

    blocks = []
    blocks.extend(instructions)
    if examples_block:
      blocks.append('')
      blocks.extend(examples_block)
    blocks.append('')
    blocks.extend(eval_block)
    if schema_block:
      blocks.append('')
      blocks.extend(schema_block)

    return '\n'.join(blocks)


In [ ]:
import io
from contextlib import redirect_stdout
# evaluation protocol and metrics
def run_single_replicate(engine_fn, engine_kwargs: dict, csv_path): # engine_kwargs: dict or args for engine_fn e.g., (X, w, a, k, b, t, esp)
    n_row_threshold = 200
    subset_n = 50
    decimals = 3
    k_smd = 10

    # generate dataset D with sample size n
    X, A, Y0, Y1, Y = engine_fn(**engine_kwargs)

    # compute ground-truth ATE
    tau_true = float(np.mean(Y1 - Y0))
    X = np.asarray(X).T
    A = np.asarray(A).reshape(-1)
    Y = np.asarray(Y).reshape(-1)

    n = X.shape[0]
    p = X.shape[1]

    df = pd.DataFrame(X, columns = [f'X{i + 1}' for i in  range(p)])
    df.insert(0, 'Y', Y)
    df.insert(0, 'A', A)

    for c in df.columns:
        if c != 'A':
            df[c] = pd.to_numeric(df[c], errors = 'coerce').round(decimals)
    df['A'] = pd.to_numeric(df['A'], errors = 'coerce').round().astype('Int64').clip(0, 1)

    df.to_csv(csv_path, index = False)
    df_clean = preclean(csv_path)
    df_clean.to_csv(csv_path, index = False)

    # serialize D into text (format A or B)
    buf = io.StringIO()
    with redirect_stdout(buf):
        if n <= n_row_threshold:
            small_to_text(csv_path)
        else:
            large_to_subset(csv_path, subset_n = subset_n)
            smd_results = summary_stats(csv_path)
            print('COVARIATE_BALANCE_TOPK')
            print(f'k = {k_smd}')
            print('columns: covariate, mean_A0, mean_A1, sd_A0, sd_A1, SMD')
            topk = smd_results[:k_smd]
            for (col, mean_A0, mean_A1, sd_A0, sd_A1, smd) in topk:
                print(
                    f'{col}: {mean_A0: .{decimals}f}, {mean_A1: .{decimals}f},'
                    f'{sd_A0: .{decimals}f}, {sd_A1: .{decimals}f}, {smd: .{decimals}f}'
                )
            print('END')
    serialized_data = buf.getvalue()
    print(serialized_data)


    # query the LLM under a fixed ICL condition
    prompt = build_prompt(serialized_data)
    print('\n' + 'PROMPT START' + '\n')
    print(prompt)
    print('\n' + 'PROMPT END' + '\n')
    raw_output = input('Run the prompt in ChatGPT and paste the LLM JSON output here, then press Enter:\n')

    # parse outputs and overlap warning flags
    format_failure = False
    parse_error = False
    ate_hat = L = U = overlap_warning = None
    parsed = None

    try:
        parsed = json.loads(raw_output)
        ate_hat = float(parsed['ate_hat'])
        L = float(parsed['ci95'][0])
        U = float(parsed['ci95'][1])
        overlap_warning = bool(parsed['overlap_warning'])

        if not (L <= ate_hat <= U):
            raise ValueError('Constraint violate ci95[0] <= ate_hat <= ci95[1]')
    except Exception as e:
        format_failure = True
        parse_error = str(e)

    return{
        'csv_path': csv_path, 'n': n, 'p':p, 'tau_true': tau_true, 'ate_hat': ate_hat,
        'ci95_lower': L, 'ci95_upper': U, 'overlap_warning': overlap_warning, 'format_failure': format_failure,
        'parse_error': parse_error#, 'raw_output': raw_output, 'prompt': prompt, 'serialized_data': serialized_data,
        #'engine_kwargs': engine_kwargs
    }


In [ ]:
results = run_single_replicate(
    main_1,
    engine_kwargs={'X':X, 'w':w, 'a':1, 'k':1, 'b':b, 't':0.5, 'esp':esp},
    csv_path="test.csv")
results

GROUP_SUMMARIES_START
group n mean_Y sd_Y
treated 712 0.6997766853932584 1.4365258330596735
control 288 -0.51665625 1.360257124915781
GROUP_SUMMARIES_END

PROMPT START

TASK_INSTRUCTIONS
You are given observational data about covariates X, treatment A (0/1), and outcome Y.
Estimate the Average Treatment Effect (ATE).
Assume unconfoundedness given X, and discuss overlap/positivity based on the provided info.
You MUST NOT fabricate numbers. Use only the provided EVALUATION_DATASET block.
Return exactly one JSON object (no extra text).
PROTOCOL_A: Difference-in-Means from GROUP_SUMMARIES
You will be given treated and control group summaries: n, mean_Y, sd_Y.
Compute:
  ate_hat = mean_Y_treated - mean_Y_control
  se_hat = sqrt(sd_Y_treated^2/n_treated + sd_Y_control^2/n_control)
  ci_95  = [ate_hat - 1.96*se_hat, ate_hat + 1.96*se_hat]
Rules:
  - If any sd_Y or n is missing, then set se_hat and ci_95 to null.
  - If no propensity information is provided, then overlap_assessment.status='unk

In [ ]:
# point estimation metrics
def point_estimation(results):
    tau_true = np.array(results['tau_true'])
    tau_hat = np.array(results['ate_hat'])
    bias = np.mean(tau_hat - tau_true)
    rmse = np.sqrt(np.mean((tau_hat - tau_true) ** 2))
    return bias, rmse

# uncertainty calibration
def compute_uncertainty(results):
    tau_true = np.array(results['tau_true'])
    L = np.array(results['ci95_lower'])
    U = np.array(results['ci95_upper'])
    coverage = np.mean((tau_true >= L) & (tau_true <= U))
    avg_width = np.mean(U - L)
    return coverage, avg_width

In [ ]:
# Test
print(point_estimation(results))
print(compute_uncertainty(results))

(np.float64(0.02100000000000002), np.float64(0.02100000000000002))
(np.float64(1.0), np.float64(0.21800000000000003))


In [ ]:
# ICL stress-tests via traps

# Trap 1: post-treatment Z adjustment temptation
def trap_1(csv_path):
  df = pd.read_csv(csv_path)
  x_cols = [c for c in df.columns if c.startswith('X')]
  p = len(x_cols)
  no = len(df)

  lambd = 1.0 # medium-size effect of treatment on Z
  n = rng.normal(loc = 0.0, scale = 0.2, size = p) # medium-size correlation of covariates on Z
  noise = rng.normal(loc = 0.0, scale = 0.2, size = no) # noise

  df['Z'] = (lambd*df['A'].values) + (n.T@df[x_cols].values) + noise

  return df

# Trap 2: include covariate V that highly predicts A but does not effect Y
def trap_2(csv_path):
  df = pd.read_csv(csv_path)
  rho = 2.0

  x_cols = [c for c in df.columns if c.startswith('X')]
  n = df.shape[0]

  V = rng.normal(loc = 0, scale = 1, size = n)
  df["V"] = V

  # ASSUMPTION: all covariates carry equal weight
  beta = np.ones(len(x_cols)) / np.sqrt(len(x_cols))
  X_matrix = df[x_cols].values
  linear_score = X_matrix @ beta + rho * V

  A_new = rng.binomial(1,sigm(linear_score))

  df["A"] = A_new

  return df

# Trap 3: weak overlap should trigger warnings and require wider intervals
def trap_3():
  ep = 0.01
  return main_5(X, w, a, k, ep, t, engine, b = None)

def check_warning(llm_output, weak_overlap):
    return llm_output["overlap_warning"] == weak_overlap

def check_stabilization_suggestion(llm_output):
    text = llm_output["notes"].lower()

    keywords = [
        "trim",
        "trimming",
        "stabilized",
        "stabilization",
        "weight truncation",
        "overlap problem"
    ]

    return any(k in text for k in keywords)

def interval_width(llm_output):
  ci95 = llm_output["ci95"]
  return ci95[1] - ci95[0]